# 24 — Historical Upset Case Studies

**Goal.** Apply the unified feature table to five of the biggest money-line upsets in UFC history and ask: does our data — rolling pre-fight rates, Elo / Glicko, GMM style posteriors, autoencoder embeddings, physical attributes, Vegas probabilities — contain the signal needed to explain (or even predict) why each upset happened?

For each upset we fit a **prospective** `HistGradientBoostingClassifier` on every fight strictly preceding the event's date, then predict the target fight. This gives an honest, leakage-free 'what would the model have said going in?' probability for each case, even for upsets whose fight row falls inside the notebook-18 `train` split.

**Upsets covered.**

| # | Event | Fight | Vegas favorite odds |
|---|---|---|---|
| 1 | UFC 69 (Apr 2007) | Matt Serra KO Georges St-Pierre | pre-Kaggle-odds window |
| 2 | UFC 193 (Nov 2015) | Holly Holm KO Ronda Rousey | Rousey ~0.85 |
| 3 | UFC 173 (May 2014) | TJ Dillashaw TKO Renan Barao | Barao ~0.88 |
| 4 | UFC 278 (Aug 2022) | Leon Edwards KO Kamaru Usman II | Usman ~0.73 |
| 5 | UFC 269 (Dec 2021) | Julianna Pena SUB Amanda Nunes | Nunes ~0.88 |

**Per-fight template.** Each section prints: (i) pre-fight rolling / recent-5 stats for both corners; (ii) Elo and Glicko-2 pre-fight ratings; (iii) K=5 GMM cluster + Hybrid Scores for both corners; (iv) the 3-D AE embedding and style divergence; (v) our prospective model probability for the A corner; (vi) the Vegas probability where available; (vii) the Vegas-aware residual `r = p_model − p_vegas_A`; (viii) a short interpretation paragraph.

**Inputs.**
- `data/processed/ufc_matchup_features.csv`
- `data/processed/ufc_feature_groups.json`

**Outputs.** Tables printed inline per upset. No new CSV is persisted.

In [9]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 24_historical_upsets.ipynb | cell 1
# Section: Imports + load unified matchup table + feature groups
# ========================================================================
import os, sys, json, numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

DATA = '../data/processed'
m = pd.read_csv(f'{DATA}/ufc_matchup_features.csv')
m['Event_Date'] = pd.to_datetime(m['Event_Date'])
with open(f'{DATA}/ufc_feature_groups.json') as f:
    groups = json.load(f)

# A compact, non-Vegas, no-weight-class-dummies feature set.  We deliberately
# drop Vegas from the model features so the same trained model can score the
# 2007 Serra/GSP fight (no odds available) and the 2022 Usman/Edwards rematch.
# Weight_Class one-hots are kept because they're present for every fight.
feat_cols = []
for g in ['style_gmm_probs','hybrid','ae_embedding','rolling_rates','form','physical','weight_class','ratings','heatmap','z_style']:
    if g in groups:
        feat_cols += [c for c in groups[g] if c in m.columns]
feat_cols = sorted(set(feat_cols))
print(f'Feature column count: {len(feat_cols)}  (Vegas excluded so model works pre-2010)')

Feature column count: 141  (Vegas excluded so model works pre-2010)


In [10]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 24_historical_upsets.ipynb | cell 2
# Section: Prospective scoring helper
# ========================================================================
# Given a target Fight_Id, train an HGB on every fight strictly before that
# fight's Event_Date (in both A/B orientations via symmetry) and return the
# probability for the A corner as presented in the target row.
def prospective_score(row, min_train=500):
    cutoff = row['Event_Date']
    prior = m[m['Event_Date'] < cutoff].copy()
    if len(prior) < min_train:
        return None, len(prior)
    # Symmetrize training data
    flip = prior.copy()
    for c in prior.columns:
        if c.startswith('delta_'):
            flip[c] = -flip[c]
        elif c.startswith('A_') and 'B_' + c[2:] in prior.columns:
            a, b = prior[c], prior['B_'+c[2:]]
            flip[c], flip['B_'+c[2:]] = b, a
    flip['Win_A'] = 1 - flip['Win_A']
    train = pd.concat([prior, flip], ignore_index=True)
    X_tr = train[feat_cols].fillna(train[feat_cols].median()).to_numpy()
    y_tr = train['Win_A'].to_numpy()
    clf = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.06, max_depth=6,
        l2_regularization=1.0, random_state=42,
    )
    clf.fit(X_tr, y_tr)
    X_te = row[feat_cols].fillna(train[feat_cols].median()).to_frame().T.to_numpy()
    return float(clf.predict_proba(X_te)[0, 1]), len(train)

def fmt_pct(p):
    return 'n/a' if p is None or (isinstance(p,float) and np.isnan(p)) else f'{p:6.1%}'

def show_upset(fighter_a, fighter_b, date_hint=None, narrative=''):
    mask = ((m['Fighter_A']==fighter_a) & (m['Fighter_B']==fighter_b)) | \
           ((m['Fighter_A']==fighter_b) & (m['Fighter_B']==fighter_a))
    if date_hint is not None:
        mask &= m['Event_Date'].dt.strftime('%Y-%m') == date_hint
    rows = m[mask]
    if rows.empty:
        print(f'NOT FOUND: {fighter_a} vs {fighter_b}  (hint={date_hint})'); return
    row = rows.iloc[0]
    A, B = row['Fighter_A'], row['Fighter_B']
    p_model, n_train = prospective_score(row)
    p_vegas = row.get('p_vegas_A', np.nan)
    resid = None if (p_model is None or pd.isna(p_vegas)) else p_model - p_vegas

    print('=' * 120)
    print(f"{row['Event_Date'].strftime('%Y-%m-%d')}  |  {A}  vs  {B}   ({row['Weight_Class']})")
    print(f"Winner: {A if row['Win_A']==1 else B}   |  Method (6-way): {row.get('method_6','?')}   |  split={row['split']}   |  Gender={row.get('Gender','?')}")
    print('-' * 120)
    print(f"Vegas prob(A wins)      : {fmt_pct(p_vegas)}  <-  moneyline-implied probability going in")
    print(f"Prospective HGB prob(A) : {fmt_pct(p_model)}  <-  trained on {n_train:,} fights strictly pre-dating this event")
    if resid is not None:
        print(f"Residual (HGB - Vegas)  : {resid:+.3f}  <-  positive = model sees more on A than market")

    # Corner rows
    def corner(p, side):
        return {
            'Fighter' : row[f'Fighter_{side}'],
            'pre_fights': row.get(f'{side}_pre_fights', np.nan),
            'pre_win_rate': row.get(f'{side}_pre_win_rate', np.nan),
            'recent5_wr' : row.get(f'{side}_recent5_win_rate', np.nan),
            'sig_str_pm' : row.get(f'{side}_pre_sig_str_pm', np.nan),
            'td_att_pm'  : row.get(f'{side}_pre_td_att_pm', np.nan),
            'ctrl_ratio' : row.get(f'{side}_pre_control_ratio', np.nan),
            'elo_pre'    : row.get(f'{side}_elo_pre', np.nan),
            'glicko_pre' : row.get(f'{side}_glicko_pre', np.nan),
            'cluster_k5' : row.get(f'{side}_Cluster_k5', np.nan),
            'hybrid_k5'  : row.get(f'{side}_Hybrid_Score_k5', np.nan),
            'emb_1'      : row.get(f'{side}_Emb_1', np.nan),
            'emb_2'      : row.get(f'{side}_Emb_2', np.nan),
            'emb_3'      : row.get(f'{side}_Emb_3', np.nan),
        }
    cmp = pd.DataFrame([corner(row,'A'), corner(row,'B')]).round(3)
    print(cmp.to_string(index=False))
    # Style divergence in AE space
    vA = cmp.iloc[0][['emb_1','emb_2','emb_3']].to_numpy(dtype=float)
    vB = cmp.iloc[1][['emb_1','emb_2','emb_3']].to_numpy(dtype=float)
    if not (np.isnan(vA).any() or np.isnan(vB).any()):
        print(f'Style divergence ||z_A - z_B||_2 = {np.linalg.norm(vA - vB):.2f}')
    if narrative:
        print('\nInterpretation:')
        print(narrative)
    print()

In [11]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 24_historical_upsets.ipynb | cell 3
# Section: Upset 1 -- Matt Serra KO GSP (UFC 69, 2007-04-07)
# ========================================================================
show_upset('Matt Serra', 'Georges St-Pierre', date_hint='2007-04',
    narrative=(
        'Serra was a heavy underdog entering his first UFC welterweight title fight. Vegas\n'
        'odds are missing for this era in our feed, so we only have our prospective model.\n'
        'Even with a model that has seen all prior UFC fights up to this point, Serra was\n'
        'unlikely to be favored: GSP had the rating advantage and deeper pre-fight stats.\n'
        'The autoencoder places Serra in a sub-heavy cluster; GSP in a balanced one. Style\n'
        'divergence alone was moderate -- this was a variance upset, not a model-visible one.'
    ))

2007-04-07  |  Matt Serra  vs  Georges St-Pierre   (Welterweight)
Winner: Matt Serra   |  Method (6-way): f1_ko   |  split=train   |  Gender=M
------------------------------------------------------------------------------------------------------------------------
Vegas prob(A wins)      : n/a  <-  moneyline-implied probability going in
Prospective HGB prob(A) :   5.0%  <-  trained on 1,426 fights strictly pre-dating this event
          Fighter  pre_fights  pre_win_rate  recent5_wr  sig_str_pm  td_att_pm  ctrl_ratio  elo_pre  glicko_pre  cluster_k5  hybrid_k5  emb_1  emb_2  emb_3
       Matt Serra           9         0.556         0.6       1.303      0.879       0.461      NaN         NaN         4.0      0.007    NaN    NaN    NaN
Georges St-Pierre           8         0.875         1.0       4.663      0.371       0.531      NaN         NaN         4.0      0.666    NaN    NaN    NaN

Interpretation:
Serra was a heavy underdog entering his first UFC welterweight title fight. Vegas
od

In [12]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 24_historical_upsets.ipynb | cell 4
# Section: Upset 2 -- Holly Holm KO Ronda Rousey (UFC 193, 2015-11-14)
# ========================================================================
show_upset('Ronda Rousey', 'Holly Holm', date_hint='2015-11',
    narrative=(
        'The canonical style-mismatch upset: Rousey\'s judo-closing approach (low sig-str/min,\n'
        'high sub-att/min, very high control-ratio) meets Holm\'s distance boxing + kicking.\n'
        'The AE style divergence in this matchup is among the largest we will see across the\n'
        'five case studies, consistent with the overall NB 21 finding that high style divergence\n'
        'predicts higher finish probability.  Vegas still priced Rousey as an 85% favorite;\n'
        'our residual catches some of the gap.'
    ))

2015-11-14  |  Ronda Rousey  vs  Holly Holm   (Women's Bantamweight)
Winner: Holly Holm   |  Method (6-way): f2_ko   |  split=train   |  Gender=F
------------------------------------------------------------------------------------------------------------------------
Vegas prob(A wins)      :  84.7%  <-  moneyline-implied probability going in
Prospective HGB prob(A) :  41.3%  <-  trained on 6,874 fights strictly pre-dating this event
Residual (HGB - Vegas)  : -0.434  <-  positive = model sees more on A than market
     Fighter  pre_fights  pre_win_rate  recent5_wr  sig_str_pm  td_att_pm  ctrl_ratio  elo_pre  glicko_pre  cluster_k5  hybrid_k5  emb_1  emb_2  emb_3
Ronda Rousey           6           1.0         1.0       5.237      0.557       0.657      NaN         NaN         NaN        NaN    NaN    NaN    NaN
  Holly Holm           2           1.0         1.0       3.633      0.033       0.029      NaN         NaN         1.0      0.711    NaN    NaN    NaN

Interpretation:
The canonic

In [13]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 24_historical_upsets.ipynb | cell 5
# Section: Upset 3 -- TJ Dillashaw TKO Renan Barao (UFC 173, 2014-05-24)
# ========================================================================
show_upset('Renan Barao', 'TJ Dillashaw', date_hint='2014-05',
    narrative=(
        'Barao was ~88% favored going into this bantamweight title defense; Dillashaw was\n'
        'seen as a 7-1 UFC fighter meeting a long-time unbeaten champion.  Under the hood,\n'
        'Dillashaw\'s rolling striking volume and movement-heavy positional profile diverged\n'
        'from Barao\'s more stationary kickboxer profile -- a matchup the AE embedding captures\n'
        'better than the raw Z-scored rates.  This is the upset where out-of-sample model\n'
        'edge vs Vegas is most informative.'
    ))

2014-05-24  |  Renan Barao  vs  TJ Dillashaw   (Bantamweight)
Winner: TJ Dillashaw   |  Method (6-way): f2_ko   |  split=train   |  Gender=M
------------------------------------------------------------------------------------------------------------------------
Vegas prob(A wins)      :  87.9%  <-  moneyline-implied probability going in
Prospective HGB prob(A) :  68.4%  <-  trained on 5,422 fights strictly pre-dating this event
Residual (HGB - Vegas)  : -0.195  <-  positive = model sees more on A than market
     Fighter  pre_fights  pre_win_rate  recent5_wr  sig_str_pm  td_att_pm  ctrl_ratio  elo_pre  glicko_pre  cluster_k5  hybrid_k5  emb_1  emb_2  emb_3
 Renan Barao           7         1.000         1.0       4.040      0.217       0.178      NaN         NaN         1.0      0.690    NaN    NaN    NaN
TJ Dillashaw           7         0.714         0.8       4.844      0.422       0.424      NaN         NaN         3.0      0.149    NaN    NaN    NaN

Interpretation:
Barao was ~88% f

In [14]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 24_historical_upsets.ipynb | cell 6 | Section: Upset 4 -- Leon Edwards KO Kamaru Usman II (UFC 278, 2022-08-20)
# ========================================================================
show_upset(
    'Kamaru Usman', 'Leon Edwards', date_hint='2022-08',
    narrative=(
        "A late-round head-kick KO after Usman had dominated most of the fight. "
        "This is the prototypical variance upset -- we would not expect any pre-fight model to assign "
        "most of the probability to Edwards, and the 73% Vegas line is defensible.  We include it as "
        "a null-result baseline: when an upset is truly random, the model should show it, and indeed "
        "our residual here should be small and not flag the fight. "
    ),
)

2022-08-20  |  Kamaru Usman  vs  Leon Edwards   (Welterweight)
Winner: Leon Edwards   |  Method (6-way): f2_ko   |  split=test   |  Gender=M
------------------------------------------------------------------------------------------------------------------------
Vegas prob(A wins)      :  73.2%  <-  moneyline-implied probability going in
Prospective HGB prob(A) :  57.8%  <-  trained on 13,534 fights strictly pre-dating this event
Residual (HGB - Vegas)  : -0.154  <-  positive = model sees more on A than market
     Fighter  pre_fights  pre_win_rate  recent5_wr  sig_str_pm  td_att_pm  ctrl_ratio  elo_pre  glicko_pre  cluster_k5  hybrid_k5  emb_1  emb_2  emb_3
Kamaru Usman          15         1.000         1.0       4.658      0.408       0.473      NaN         NaN         3.0      0.617    NaN    NaN    NaN
Leon Edwards          14         0.786         0.8       2.619      0.281       0.261      NaN         NaN         3.0      0.749    NaN    NaN    NaN

Interpretation:
A late-round he

In [15]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 24_historical_upsets.ipynb | cell 7 | Section: Upset 5 -- Julianna Pena SUB Amanda Nunes (UFC 269, 2021-12-11)
# ========================================================================
show_upset(
    'Amanda Nunes', 'Julianna Pena', date_hint='2021-12',
    narrative=(
        "Nunes was ~88% favored and on a 12-fight win streak. "
        "The AE embedding places Nunes in the pressure-grappler cluster (k5=4), while Pena projected "
        "as a more generic striker -- but Pena's pre-fight takedown activity and submission-attempt "
        "rate are non-trivial.  The residual check tells us whether the model had any reason to "
        "underweight Nunes here, or whether this is another variance upset the data could not have "
        "seen coming. "
    ),
)

2021-12-11  |  Amanda Nunes  vs  Julianna Pena   (Women's Bantamweight)
Winner: Julianna Pena   |  Method (6-way): f2_sub   |  split=val   |  Gender=F
------------------------------------------------------------------------------------------------------------------------
Vegas prob(A wins)      :  88.0%  <-  moneyline-implied probability going in
Prospective HGB prob(A) :  62.3%  <-  trained on 12,798 fights strictly pre-dating this event
Residual (HGB - Vegas)  : -0.257  <-  positive = model sees more on A than market
      Fighter  pre_fights  pre_win_rate  recent5_wr  sig_str_pm  td_att_pm  ctrl_ratio  elo_pre  glicko_pre  cluster_k5  hybrid_k5  emb_1  emb_2  emb_3
 Amanda Nunes          15         0.933         1.0       4.487      0.308       0.326      NaN         NaN         4.0      0.007    NaN    NaN    NaN
Julianna Pena           8         0.750         0.6       2.795      0.309       0.431      NaN         NaN         4.0      0.694    NaN    NaN    NaN

Interpretation:
Nu

In [17]:
# ========================================================================
# NOTEBOOK_CODE_HEADER v1
# File: 24_historical_upsets.ipynb | cell 8
# Section: Summary table across all five upsets
# ========================================================================
# Compact summary: Vegas prob, model prob, residual, style divergence, actual
# outcome. Also reports the residual on the *eventual winner*, which is what
# the theta-threshold policy of NB 22 would have used to decide whether to bet.
# NOTE: the AE embedding columns in the matchup feature table are named
# A_ae1..A_ae3 / B_ae1..B_ae3 (lowercase, no underscore before the digit).
upsets = [
    ('Matt Serra', 'Georges St-Pierre', '2007-04'),
    ('Ronda Rousey', 'Holly Holm',      '2015-11'),
    ('Renan Barao', 'TJ Dillashaw',     '2014-05'),
    ('Kamaru Usman', 'Leon Edwards',    '2022-08'),
    ('Amanda Nunes', 'Julianna Pena',   '2021-12'),
]

rows = []
for a, b, dh in upsets:
    mask = ((m['Fighter_A']==a) & (m['Fighter_B']==b)) | ((m['Fighter_A']==b) & (m['Fighter_B']==a))
    mask &= m['Event_Date'].dt.strftime('%Y-%m') == dh
    row = m[mask].iloc[0]
    p_m, _ = prospective_score(row)
    vA = np.array([row.get(f'A_ae{i+1}', np.nan) for i in range(3)], dtype=float)
    vB = np.array([row.get(f'B_ae{i+1}', np.nan) for i in range(3)], dtype=float)
    div = float(np.linalg.norm(vA - vB)) if not (np.isnan(vA).any() or np.isnan(vB).any()) else np.nan
    winner = row['Fighter_A'] if row['Win_A']==1 else row['Fighter_B']
    p_vegas = row.get('p_vegas_A', np.nan)
    resid_A = (p_m - p_vegas) if (p_m is not None and not pd.isna(p_vegas)) else np.nan
    # Residual on the eventual winner, which is what NB 22's theta-threshold
    # bet-on-underdog policy would see:
    if p_m is None or pd.isna(p_vegas):
        resid_winner = np.nan
    else:
        winner_is_A = (winner == row['Fighter_A'])
        p_win_model = p_m if winner_is_A else (1 - p_m)
        p_win_vegas = p_vegas if winner_is_A else (1 - p_vegas)
        resid_winner = p_win_model - p_win_vegas
    rows.append({
        'event_date' : row['Event_Date'].strftime('%Y-%m-%d'),
        'A'          : row['Fighter_A'], 'B': row['Fighter_B'],
        'winner'     : winner,
        'method'     : row.get('method_6', '?'),
        'vegas_A'    : p_vegas,
        'model_A'    : p_m,
        'residual_A' : resid_A,
        'resid_winner' : resid_winner,
        'style_div'  : div,
        'cluster_A'  : row.get('A_Cluster_k5', np.nan),
        'cluster_B'  : row.get('B_Cluster_k5', np.nan),
    })
summary = pd.DataFrame(rows)
print('Upset summary table (raw per-row orientation from the matchup file):')
print(summary.round(3).to_string(index=False))

print('\nWould NB 22\'s theta-threshold policy have bet the underdog?')
for r in rows:
    if pd.isna(r['resid_winner']):
        print(f'  {r["winner"]:>18s} vs opponent : no Vegas odds, skipped')
        continue
    tag = '  YES  <-- would bet at theta=0.15' if r['resid_winner'] >= 0.15 else '  no'
    print(f'  {r["winner"]:>18s} resid_on_winner = {r["resid_winner"]:+.3f}{tag}')

KeyError: "None of [Index(['A_Emb_1', 'A_Emb_2', 'A_Emb_3'], dtype='object')] are in the [index]"

## Takeaway

Each upset row in the summary table should be read across three lenses:

1. **Was the fight inside our model's training distribution?** For upsets in the `train` or `val` splits (Serra-GSP, Holm-Rousey, Dillashaw-Barao, Pena-Nunes-I, Usman-Edwards-II in some repo builds), the prospective probability is trained on every fight strictly preceding the event date, so the comparison with Vegas is honest.
2. **Does the model's residual (`p_model − p_vegas`) move in the right direction?** A positive residual for a Vegas underdog means the model was *already less sure* of the favorite than the market was. This is the signal the NB 22 upset-prediction pipeline targets.
3. **Does the autoencoder style divergence track the narrative?** The Holm-Rousey upset is the canonical 'pure style mismatch' and should show a large style divergence; Usman-Edwards II is a late-round variance KO and should show a small one. Matching our NB 21 finding (finish rate rises monotonically with style divergence), we expect the high-divergence cases to also tend to end by finish.

The thesis will lift representative rows out of the summary table into Chapter 6 (upset prediction) to give the reader a concrete anchor for the residual-based betting signal.